# Text-Conditioned Video Generation — GPU Training

Trains both models on a free Colab/Kaggle GPU and produces the results the local
machine cannot (it has no CUDA device).

**Runtime → Change runtime type → T4 GPU** before running anything.

Total time: roughly **1.5–2 hours** for both models plus evaluation.

Everything is deterministic from the seeds in the configs, so the dataset is
regenerated here rather than uploaded — the code is a few hundred KB, the dataset is
1.5 GB.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> T4 GPU')

## 2. Get the code

Either clone from GitHub (preferred), or upload the repo as a zip and unzip it.

In [ ]:
# Option A - clone (replace with your repo URL once pushed)
# !git clone https://github.com/<you>/text_video.git
# %cd text_video

# Option B - upload a zip of the project, then:
# from google.colab import files; files.upload()
# !unzip -q text_video.zip && cd text_video

%cd /content/text_video
!ls

In [ ]:
!pip install -q open_clip_torch scikit-image opencv-python-headless imageio pyyaml
!pip install -q -e .
print('installed')

## 3. Mount Drive for checkpoints

Free sessions disconnect without warning. Checkpoints are mirrored to Drive so a
dropped session costs minutes, not the whole run — `--resume auto` picks up exactly
where it stopped, including optimizer and RNG state.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MIRROR = '/content/drive/MyDrive/text2video_checkpoints'
import os; os.makedirs(MIRROR, exist_ok=True)
print('checkpoints will be mirrored to', MIRROR)

## 4. Build the dataset and text embeddings

~2 minutes for the clips, well under a minute for the CLIP embeddings on GPU.

Expected: **20,000 train / 2,000 val / 2,000 test** clips, ~75% unique captions on
train, ~68% of clips containing a bounce.

In [ ]:
!python scripts/build_dataset.py --config configs/dataset.yaml

In [ ]:
# --verify runs linear probes proving the frozen embeddings actually carry
# direction / digit / speed. Expect ~93% / ~99% / ~68%.
!python scripts/build_text_embeddings.py --device cuda --verify

## 5. Train the independent digit classifier

The judge for the grounding metric. Shares no weights with the generators.
Expect ~98.7% held-out accuracy.

In [ ]:
!python scripts/train_digit_classifier.py

## 6. Smoke test before spending GPU hours

30 steps on 64 clips. If this fails, real training would fail an hour in.

In [ ]:
!python scripts/train.py --config configs/train_convlstm.yaml --smoke
!python scripts/train.py --config configs/train_baseline.yaml --smoke

## 7. Train the BASELINE (frame-independent)

~30–45 min on a T4. Watch `recon` fall; `kl` will rise during warm-up (beta ramps
from 0 over the first 2,000 steps) and then stabilise.

If the session drops, just re-run this cell — `--resume auto` continues.

In [ ]:
!python scripts/train.py --config configs/train_baseline.yaml \
    --device cuda --resume auto --mirror-dir $MIRROR/baseline

## 8. Train the MAIN MODEL (ConvLSTM)

Same config, same seed, same step budget — the only difference is the recurrence.
That is what makes the comparison attributable to temporal modelling.

In [ ]:
!python scripts/train.py --config configs/train_convlstm.yaml \
    --device cuda --resume auto --mirror-dir $MIRROR/convlstm

## 9. (Optional) Capacity-matched baseline

The ConvLSTM carries ~2.2M more parameters than the baseline. This run widens the
baseline's MLP to match, so "the ConvLSTM won because it had more capacity" can be
ruled out rather than argued about. Run it if you have GPU time left.

In [ ]:
!python scripts/train.py --config configs/train_baseline_wide.yaml \n    --device cuda --resume auto --mirror-dir $MIRROR/baseline_wide

## 10. Evaluate everything

Runs both models plus two reference rows: the **real-data ceiling** (what the metrics
top out at on ground truth) and the **static control** (frame 0 repeated 16×, which
scores a perfect SSIM while generating no motion — the reason SSIM is never reported
alone).

In [ ]:
!python scripts/evaluate.py --all --device cuda --num-clips 500 --split test

In [ ]:
!python scripts/build_report.py

## 11. Qualitative samples

In [ ]:
!python scripts/sample.py --compare --device cuda -n 2

from IPython.display import Image, display
for variant in ('baseline', 'convlstm'):
    path = f'outputs/samples/{variant}/samples.png'
    if os.path.exists(path):
        print(variant)
        display(Image(path))

## 12. Package the results to bring back

Downloads a zip with checkpoints, evaluation records, the results table and sample
images. Unzip it into the project root locally to continue.

In [ ]:
!mkdir -p results_bundle
!cp -r outputs/eval results_bundle/ 2>/dev/null || true
!cp -r outputs/samples results_bundle/ 2>/dev/null || true
!cp RESULTS.md results_bundle/ 2>/dev/null || true
!cp outputs/digit_classifier.json results_bundle/ 2>/dev/null || true

# best checkpoint + run record per model (skip the bulky rolling checkpoints)
!for r in outputs/runs/*/; do \
    n=$(basename $r); \
    case $n in *_smoke) continue;; esac; \
    mkdir -p results_bundle/runs/$n/checkpoints; \
    cp $r/checkpoints/best.pt results_bundle/runs/$n/checkpoints/ 2>/dev/null || true; \
    cp $r/run_record.json $r/history.json results_bundle/runs/$n/ 2>/dev/null || true; \
  done

!du -sh results_bundle
!zip -qr results_bundle.zip results_bundle
!ls -lh results_bundle.zip

from google.colab import files
files.download('results_bundle.zip')

Also copy the bundle to Drive as a backup, in case the download is interrupted.

In [ ]:
!cp results_bundle.zip $MIRROR/
print('backed up to', MIRROR)